<a href="https://colab.research.google.com/github/arturexxx/Alura-AgenteIA-RAG-PDF/blob/main/notebooks/Agente_RETCC_Alura_Desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==========================================================================================================
# PROYECTO: AGENTE DE IA PARA CONSULTAS DEL REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL (RETCC)
# ==========================================================================================================
# Descripción:
# Este proyecto implementa un sistema RAG (Retrieval-Augmented
# Generation) capaz de comprender el contenido de múltiples
# documentos PDF relacionados con el Registro Nacional de
# Trabajadores de Construcción Civil (RETCC) en el Estado Peruano.
#
# El agente podrá responder preguntas sobre:
# - Reglamento del RETCC.
# - Requisitos para la inscripción del Carné RETCC.
# - Requisitos para Renovacion de Carné RETCC.
#
# Documentos utilizados:
# 1. Reglamento_Inscripcion.pdf
# 2. Requisitos_Inscripcion_Carnet_RETCC.pdf
#
# Tecnologías:
# - Python
# - Google Colab
# - LangChain
# - Hugging Face (Embeddings)
# - FAISS (Base Vectorial)
# - Google Gemini (LLM)
#
# Objetivo:
# Construir un asistente inteligente que consulte la información
# de los documentos y responda preguntas utilizando RAG.
# ============================================================

print(" Iniciando el proyecto: Agente IA RAG para documentos del RETCC")

 Iniciando el proyecto: Agente IA RAG para documentos del RETCC


In [8]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    pdfplumber \
    python-dotenv \
    gradio==5.49.1

In [75]:
from google.colab import files

archivos_subidos = files.upload()

Saving Canales_de_Atencion_Retcc_2.pdf to Canales_de_Atencion_Retcc_2.pdf
Saving Reglamento_Incripcion_RETCC.pdf to Reglamento_Incripcion_RETCC.pdf
Saving Requisitos_Inscripcion_Carnet_RETCC_2.pdf to Requisitos_Inscripcion_Carnet_RETCC_2.pdf


In [10]:
lista_pdfs = list(archivos_subidos.keys())

print(lista_pdfs)

['Canales_de_Atencion_Retcc_2.pdf', 'Reglamento_Incripcion_RETCC.pdf', 'Requisitos_Inscripcion_Carnet_RETCC_2.pdf']


In [11]:
import re
import pdfplumber
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

In [12]:
def normalizar_celda(valor) -> str:
    """
    Limpia una celda extraída desde una tabla PDF.
    Conserva la información, pero elimina cortes de línea innecesarios.
    """
    if valor is None:
        return ""

    texto = str(valor).replace("\xa0", " ")

    # Une palabras cortadas al final de línea:
    # "regio\nnal" -> "regional"
    texto = re.sub(r"(?<=\w)\n(?=\w)", "", texto)

    # Convierte los demás saltos de línea en espacios
    texto = re.sub(r"\s*\n\s*", " ", texto)

    # Reduce espacios repetidos
    texto = re.sub(r"[ \t]+", " ", texto)

    return texto.strip()

In [13]:
def es_fila_region(fila: list) -> bool:
    """
    Identifica filas negras que contienen solo el número y nombre de región.
    """
    celdas = [normalizar_celda(celda) for celda in fila]

    texto = " ".join(celda for celda in celdas if celda)

    if not texto:
        return False

    # Una fila de región normalmente contiene un número y un nombre,
    # pero no contiene horarios, correos ni direcciones.
    tiene_numero = any(re.fullmatch(r"\d{1,2}", celda) for celda in celdas)
    tiene_horario = bool(re.search(r"\d{1,2}:\d{2}", texto))
    tiene_correo = "@" in texto
    tiene_direccion = any(
        palabra in texto.upper()
        for palabra in [
            "DIRECCIÓN REGIONAL",
            "GERENCIA REGIONAL",
            "ZONA DE TRABAJO",
            "OFICINA DE"
        ]
    )

    return tiene_numero and not tiene_horario and not tiene_correo and not tiene_direccion

In [14]:
def obtener_region_de_fila(fila: list) -> str:
    """
    Obtiene el nombre de la región desde una fila separadora.
    """
    celdas = [normalizar_celda(celda) for celda in fila]

    candidatos = [
        celda
        for celda in celdas
        if celda
        and not re.fullmatch(r"\d{1,2}", celda)
        and celda.upper() not in {
            "N°",
            "GOBIERNO",
            "REGIONAL/DIRECCIÓN U",
            "OFICINA"
        }
    ]

    if not candidatos:
        return ""

    return max(candidatos, key=len).strip().upper()

In [15]:
def detectar_columnas_tabla(fila: list) -> dict:
    """
    Detecta automáticamente las posiciones principales de la tabla.
    El PDF usa 18 columnas en las páginas 1 a 7
    y 11 columnas en la página 8.
    """
    cantidad = len(fila)

    if cantidad >= 18:
        return {
            "oficina": 3,
            "aprobador": 7,
            "contacto": 9,
            "horario": 10,
            "web": 11,
            "pago": 16
        }

    if cantidad >= 11:
        return {
            "oficina": 1,
            "aprobador": 4,
            "contacto": 7,
            "horario": 8,
            "web": 9,
            "pago": 10
        }

    return {}

In [16]:
def extraer_registros_tabla(pdf: str) -> list[Document]:
    """
    Convierte cada sede/oficina de la tabla en un Document independiente.

    Esto evita que el RAG mezcle:
    - una región con otra;
    - una sede con otra;
    - un horario con una dirección distinta.
    """
    documentos_tabla = []
    region_actual = ""

    with pdfplumber.open(pdf) as archivo_pdf:

        for numero_pagina, pagina in enumerate(
            archivo_pdf.pages,
            start=1
        ):
            tablas = pagina.extract_tables()

            if not tablas:
                continue

            # La primera tabla de cada página contiene la información principal.
            tabla = tablas[0]

            for fila in tabla:
                if not fila:
                    continue

                # Detectar cambio de región.
                if es_fila_region(fila):
                    nueva_region = obtener_region_de_fila(fila)

                    if nueva_region:
                        region_actual = nueva_region

                    continue

                columnas = detectar_columnas_tabla(fila)

                if not columnas:
                    continue

                oficina = normalizar_celda(
                    fila[columnas["oficina"]]
                )

                aprobador = normalizar_celda(
                    fila[columnas["aprobador"]]
                )

                contacto = normalizar_celda(
                    fila[columnas["contacto"]]
                )

                horario = normalizar_celda(
                    fila[columnas["horario"]]
                )

                web = normalizar_celda(
                    fila[columnas["web"]]
                )

                pago = normalizar_celda(
                    fila[columnas["pago"]]
                )

                # Ignorar cabeceras y filas vacías.
                if not oficina:
                    continue

                if "GOBIERNO REGIONAL" in oficina.upper():
                    continue

                if oficina.upper() in {
                    "REGIONAL/DIRECCIÓN U OFICINA",
                    "OFICINA"
                }:
                    continue

                contenido = f"""
REGIÓN: {region_actual}
SEDE U OFICINA: {oficina}
SERVIDOR PÚBLICO APROBADOR: {aprobador or "No indica"}
CORREO Y/O TELÉFONO: {contacto or "No indica"}
HORARIO DE ATENCIÓN: {horario or "No indica"}
PÁGINA WEB U OTROS: {web or "No indica"}
PAGO POR DUPLICADO: {pago or "No indica"}
                """.strip()

                documentos_tabla.append(
                    Document(
                        page_content=contenido,
                        metadata={
                            "source": pdf,
                            "page": numero_pagina - 1,
                            "pagina_real": numero_pagina,
                            "tipo_contenido": "registro_tabla",
                            "region": region_actual,
                            "oficina": oficina
                        }
                    )
                )

    return documentos_tabla

In [18]:
documentos = []
documentos_tabla = []

for pdf in lista_pdfs:
    print(f"Cargando texto normal: {pdf}")

    loader = PyPDFLoader(pdf)
    docs_pdf = loader.load()

    documentos.extend(docs_pdf)

    print(f"Extrayendo tablas estructuradas: {pdf}")

    registros_pdf = extraer_registros_tabla(pdf)
    documentos_tabla.extend(registros_pdf)

    print(
        f"  Registros estructurados encontrados: "
        f"{len(registros_pdf)}"
    )

print("\n" + "-" * 60)
print(f"Páginas normales cargadas: {len(documentos)}")
print(f"Registros de tabla creados: {len(documentos_tabla)}")

Cargando texto normal: Canales_de_Atencion_Retcc_2.pdf
Extrayendo tablas estructuradas: Canales_de_Atencion_Retcc_2.pdf
  Registros estructurados encontrados: 60
Cargando texto normal: Reglamento_Incripcion_RETCC.pdf
Extrayendo tablas estructuradas: Reglamento_Incripcion_RETCC.pdf
  Registros estructurados encontrados: 0
Cargando texto normal: Requisitos_Inscripcion_Carnet_RETCC_2.pdf
Extrayendo tablas estructuradas: Requisitos_Inscripcion_Carnet_RETCC_2.pdf
  Registros estructurados encontrados: 0

------------------------------------------------------------
Páginas normales cargadas: 17
Registros de tabla creados: 60


In [19]:
import re

def limpiar_texto(texto: str) -> str:
    """
    Limpia texto narrativo extraído del PDF.

    Los registros de tabla no pasan por esta función porque ya fueron
    convertidos a una estructura clara, una sede por documento.
    """

    texto = texto.replace("\xa0", " ")

    # Une palabras cortadas con guion al final de línea.
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # Une palabras partidas por salto de línea sin guion.
    texto = re.sub(r"(?<=\w)\n(?=\w)", "", texto)

    # Conserva párrafos, pero reemplaza saltos simples.
    texto = re.sub(r"(?<!\n)\n(?!\n)", " ", texto)

    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r"[ \t]+", " ", texto)

    return texto.strip()


print("Función de limpieza creada correctamente.")

Función de limpieza creada correctamente.


In [20]:
documentos_limpios = []

# Limpiar únicamente las páginas normales.
for documento in documentos:
    documento.page_content = limpiar_texto(
        documento.page_content
    )

    documento.metadata["tipo_contenido"] = "pagina_pdf"

    documentos_limpios.append(documento)

# Agregar los registros estructurados sin volver a limpiarlos.
documentos_limpios.extend(documentos_tabla)

print(
    f"Documentos preparados: {len(documentos_limpios)}"
)

print(
    f"  Páginas normales: {len(documentos)}"
)

print(
    f"  Registros de tabla: {len(documentos_tabla)}"
)

Documentos preparados: 77
  Páginas normales: 17
  Registros de tabla: 60


In [21]:
print(documentos_limpios[0].page_content[:2000])

"Decenio de la igualdad de oportunidades para mujeres y hombres" "Año de la recuperación y consolidación de la economía peruana" REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL – RETCC CANALES DE ATENCIÓN OFICIALES, POR CADA DIRECCIÓN Y/O GERENCIA REGIONAL DE TRABAJO Y PROMOCIÓN DEL EMPLEO Modalidades de atención: 1. Presencial: Ver las direcciones consignadas en el cuadro 2. Virtual: A través del siguiente link: https://portal.trabajo.gob.pe/retcc-virtual/ Nota 1: Plazo de atención máximo es 3 días hábiles. Vencido dicho plazo, puede acercarse al centro de atención seleccionado a exigir la evaluación de su solicitud. Nota 2: En caso de observarse la solicitud, el plazo de subsanación es de 48 horas. Nota 3: El trabajador puede solicitar la renovación dentro de los 45 días calendario anteriores a la caducidad de la vigencia de su carné de construcción civil. Seguimiento de trámite: A través del siguiente link: https://portal.trabajo.gob.pe/si.retccseg/app/#/ RECUERDA QUE EL REC

In [22]:
documentos_limpios = [
    documento
    for documento in documentos_limpios
    if documento.page_content.strip()
]

print(f"Páginas con contenido: {len(documentos_limpios)}")

Páginas con contenido: 77


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ]
)

print("Divisor configurado correctamente.")

Divisor configurado correctamente.


In [24]:
# =========================================================
# DIVISIÓN HÍBRIDA
# =========================================================
# Las páginas normales se dividen en fragmentos.
# Cada fila de la tabla se conserva completa para no mezclar sedes.

paginas_normales = [
    documento
    for documento in documentos_limpios
    if documento.metadata.get("tipo_contenido") == "pagina_pdf"
]

registros_tabla = [
    documento
    for documento in documentos_limpios
    if documento.metadata.get("tipo_contenido") == "registro_tabla"
]

fragmentos_normales = text_splitter.split_documents(
    paginas_normales
)

fragmentos = fragmentos_normales + registros_tabla

print(f"Páginas normales procesadas: {len(paginas_normales)}")
print(f"Fragmentos narrativos: {len(fragmentos_normales)}")
print(f"Registros completos de tabla: {len(registros_tabla)}")
print(f"Total enviado a FAISS: {len(fragmentos)}")

Páginas normales procesadas: 17
Fragmentos narrativos: 58
Registros completos de tabla: 60
Total enviado a FAISS: 118


In [25]:
for indice, fragmento in enumerate(fragmentos):
    pagina_langchain = fragmento.metadata.get("page", 0)

    fragmento.metadata["chunk_id"] = indice
    fragmento.metadata["pagina_real"] = fragmento.metadata.get(
        "pagina_real",
        pagina_langchain + 1
    )

print("Metadatos de trazabilidad agregados.")

Metadatos de trazabilidad agregados.


In [26]:
cantidad_mostrar = min(5, len(fragmentos))

for indice in range(cantidad_mostrar):
    fragmento = fragmentos[indice]

    print("=" * 80)
    print(f"Fragmento: {fragmento.metadata['chunk_id']}")
    print(f"Página: {fragmento.metadata['pagina_real']}")
    print(f"Caracteres: {len(fragmento.page_content)}")
    print("-" * 80)
    print(fragmento.page_content[:700])
    print()

Fragmento: 0
Página: 1
Caracteres: 877
--------------------------------------------------------------------------------
"Decenio de la igualdad de oportunidades para mujeres y hombres" "Año de la recuperación y consolidación de la economía peruana" REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL – RETCC CANALES DE ATENCIÓN OFICIALES, POR CADA DIRECCIÓN Y/O GERENCIA REGIONAL DE TRABAJO Y PROMOCIÓN DEL EMPLEO Modalidades de atención: 1. Presencial: Ver las direcciones consignadas en el cuadro 2. Virtual: A través del siguiente link: https://portal.trabajo.gob.pe/retcc-virtual/ Nota 1: Plazo de atención máximo es 3 días hábiles. Vencido dicho plazo, puede acercarse al centro de atención seleccionado a exigir la evaluación de su solicitud. Nota 2: En caso de observarse la solicitud, el plazo de subsanació

Fragmento: 1
Página: 1
Caracteres: 791
--------------------------------------------------------------------------------
. Seguimiento de trámite: A través del siguiente link: htt

In [27]:
fragmentos_vacios = [
    fragmento
    for fragmento in fragmentos
    if not fragmento.page_content.strip()
]

longitudes = [
    len(fragmento.page_content)
    for fragmento in fragmentos
]

print(f"Total de fragmentos: {len(fragmentos)}")
print(f"Fragmentos vacíos: {len(fragmentos_vacios)}")

if longitudes:
    print(f"Tamaño mínimo: {min(longitudes)} caracteres")
    print(f"Tamaño máximo: {max(longitudes)} caracteres")
    print(f"Tamaño promedio: {sum(longitudes) / len(longitudes):.2f} caracteres")

Total de fragmentos: 118
Fragmentos vacíos: 0
Tamaño mínimo: 151 caracteres
Tamaño máximo: 1200 caracteres
Tamaño promedio: 683.83 caracteres


In [28]:
!pip install -qU langchain-huggingface sentence-transformers

In [29]:
from langchain_huggingface import HuggingFaceEmbeddings

print("HuggingFaceEmbeddings importado correctamente.")

HuggingFaceEmbeddings importado correctamente.


In [30]:
from langchain_huggingface import HuggingFaceEmbeddings

In [31]:
MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print(f"Modelo seleccionado: {MODELO_EMBEDDINGS}")

Modelo seleccionado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [32]:
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo seleccionado: {dispositivo}")

if dispositivo == "cuda":
    print(f"GPU disponible: {torch.cuda.get_device_name(0)}")
else:
    print("Se utilizará CPU.")

Dispositivo seleccionado: cpu
Se utilizará CPU.


In [33]:
modelo_embeddings = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDINGS,
    model_kwargs={
        "device": dispositivo
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 32
    }
)

print("Modelo de embeddings cargado correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de embeddings cargado correctamente.


In [34]:
import numpy as np

def similitud_coseno(vector_a, vector_b):
    vector_a = np.array(vector_a)
    vector_b = np.array(vector_b)

    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) *
        np.linalg.norm(vector_b)
    )

In [35]:
from langchain_community.vectorstores import FAISS

print("FAISS importado correctamente.")

FAISS importado correctamente.


In [36]:
vectorstore = FAISS.from_documents(
    documents=fragmentos,
    embedding=modelo_embeddings
)

print("Base vectorial creada correctamente.")

Base vectorial creada correctamente.


In [37]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [38]:
!pip install -qU langchain-google-genai

In [39]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("Integración de Gemini importada correctamente.")

Integración de Gemini importada correctamente.


In [40]:
from google.colab import userdata
import os

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "No se encontró GOOGLE_API_KEY en los secretos de Colab."
    )

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API Key de Gemini cargada correctamente.")

API Key de Gemini cargada correctamente.


In [41]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_retries=1
)

print("Modelo Gemini configurado correctamente.")

Modelo Gemini configurado correctamente.


In [42]:
# =========================================================
# RETRIEVER MEJORADO PARA TABLAS
# =========================================================
# MMR evita recuperar cuatro fragmentos casi idénticos.
# fetch_k examina más candidatos antes de elegir los mejores.

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 30,
        "lambda_mult": 0.75
    }
)

print("Retriever configurado para documentos narrativos y tablas.")

Retriever configurado para documentos narrativos y tablas.


In [43]:
from langchain_core.prompts import ChatPromptTemplate

prompt_rag = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Eres un asistente especializado en responder preguntas sobre
los documentos del Registro Nacional de Trabajadores de
Construcción Civil, RETCC.

Debes responder utilizando exclusivamente la información
contenida en el contexto proporcionado.

El contexto puede contener registros estructurados de una tabla.
Cada registro representa una sede u oficina independiente y puede
incluir región, dirección, aprobador, contacto, horario, página web
y pago por duplicado.

Reglas obligatorias:

1. No inventes información.
2. No utilices conocimientos externos.
3. Si la respuesta no se encuentra en el contexto, responde:
   "No encontré esa información en los documentos proporcionados."
4. Responde en español.
5. Utiliza un lenguaje claro, ordenado y formal.
6. No mezcles información de regiones o sedes diferentes.
7. Cuando una región tenga varias sedes, enumera cada sede por separado.
8. Para cada sede, relaciona únicamente su propio horario, dirección,
   contacto y costo.
9. Si un dato dice "No indica" o "No reportó", comunícalo de esa forma.
10. Cuando existan requisitos o pasos, preséntalos de manera enumerada.
11. No menciones información que no aparezca en el contexto.
12. No menciones archivos, páginas, documentos recuperados ni fuentes.
13. Responde únicamente la consulta del usuario.
            """
        ),
        (
            "human",
            """
Contexto recuperado de los documentos:

{contexto}

Pregunta del usuario:

{pregunta}
            """
        )
    ]
)

print("Prompt del RAG creado correctamente.")

Prompt del RAG creado correctamente.


In [44]:
def formatear_contexto(documentos):
    bloques = []

    for numero, documento in enumerate(documentos, start=1):
        archivo = documento.metadata.get(
            "source",
            "Documento desconocido"
        )

        pagina = documento.metadata.get(
            "pagina_real",
            documento.metadata.get("page", 0) + 1
        )

        bloque = f"""
FRAGMENTO {numero}
Archivo: {archivo}
Página: {pagina}

Contenido:
{documento.page_content}
"""

        bloques.append(bloque.strip())

    return "\n\n" + ("\n\n" + "-" * 80 + "\n\n").join(bloques)

In [45]:
def responder_pregunta(pregunta: str) -> dict:
    """
    Recupera información desde FAISS y genera una respuesta
    mediante Gemini utilizando exclusivamente los PDF.
    """
    if not pregunta or not pregunta.strip():
        return {
            "respuesta": "Debe ingresar una pregunta.",
            "fuentes": []
        }

    pregunta = pregunta.strip()

    # 1. Recuperar fragmentos desde FAISS
    documentos = retriever.invoke(pregunta)

    # 2. Preparar el contexto
    contexto = formatear_contexto(documentos)

    # 3. Construir el prompt
    mensajes = prompt_rag.invoke(
        {
            "contexto": contexto,
            "pregunta": pregunta
        }
    )

    # 4. Enviar el prompt a Gemini
    resultado = llm.invoke(mensajes)

    # 5. Preparar las fuentes sin duplicados
    fuentes = []

    for documento in documentos:
        fuente = {
            "archivo": documento.metadata.get(
                "source",
                "Documento desconocido"
            ),
            "pagina": documento.metadata.get(
                "pagina_real",
                documento.metadata.get("page", 0) + 1
            )
        }

        if fuente not in fuentes:
            fuentes.append(fuente)

    return {
        "respuesta": resultado.content,
        "fuentes": fuentes,
        "documentos": documentos
    }

In [46]:
def preguntar_al_agente(pregunta: str) -> None:
    resultado = responder_pregunta(pregunta)

    print("=" * 90)
    print("PREGUNTA")
    print("=" * 90)
    print(pregunta)

    print("\n" + "=" * 90)
    print("RESPUESTA")
    print("=" * 90)
    print(resultado["respuesta"])

    print("\n" + "=" * 90)
    print("FUENTES RECUPERADAS")
    print("=" * 90)

    if resultado["fuentes"]:
        for fuente in resultado["fuentes"]:
            print(
                f"- {fuente['archivo']}, "
                f"página {fuente['pagina']}"
            )
    else:
        print("No se recuperaron fuentes.")

In [47]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

print("Componentes de memoria importados correctamente.")


Componentes de memoria importados correctamente.


In [48]:
almacen_conversaciones = {}

def obtener_historial(session_id: str) -> InMemoryChatMessageHistory:
    """
    Recupera el historial de una sesión.
    Si no existe, crea uno nuevo.
    """

    if session_id not in almacen_conversaciones:
        almacen_conversaciones[session_id] = InMemoryChatMessageHistory()

    return almacen_conversaciones[session_id]


print("Almacén de conversaciones creado correctamente.")

Almacén de conversaciones creado correctamente.


In [49]:
prompt_reformular = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Tu tarea es reformular la última pregunta del usuario como una
            pregunta independiente y completa, utilizando el historial de
            conversación.

            Reglas:

            1. No respondas la pregunta.
            2. Devuelve únicamente la pregunta reformulada.
            3. Conserva el idioma español.
            4. No agregues información que no aparezca en la conversación.
            5. Si la pregunta ya es independiente, devuélvela sin modificarla.

            Ejemplo:

            Historial:
            Usuario: ¿Cuáles son los requisitos para inscribirse en el RETCC?

            Última pregunta:
            ¿Y cuánto demora?

            Pregunta reformulada:
            ¿Cuánto demora el trámite de inscripción en el RETCC?
            """
        ),
        MessagesPlaceholder(variable_name="historial"),
        (
            "human",
            "{pregunta}"
        )
    ]
)

print("Prompt de reformulación creado correctamente.")

Prompt de reformulación creado correctamente.


In [50]:
def reformular_pregunta(
    pregunta: str,
    historial: list
) -> str:
    """
    Convierte una pregunta continuada en una pregunta independiente.
    """

    # Si no existe historial, no es necesario reformular
    if not historial:
        return pregunta.strip()

    mensajes = prompt_reformular.invoke(
        {
            "historial": historial,
            "pregunta": pregunta.strip()
        }
    )

    resultado = llm.invoke(mensajes)

    pregunta_reformulada = resultado.content.strip()

    return pregunta_reformulada

In [51]:
prompt_rag_con_memoria = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Eres un asistente especializado en los documentos del Registro
            Nacional de Trabajadores de Construcción Civil, RETCC.

            Debes responder utilizando exclusivamente la información contenida
            en el contexto recuperado de los documentos.

            El contexto puede incluir registros estructurados de una tabla.
            Cada registro corresponde a una sede específica y contiene campos
            como región, sede u oficina, dirección, aprobador, contacto, horario,
            página web y pago por duplicado.

            Reglas obligatorias:

            1. No inventes información.
            2. No uses conocimientos externos.
            3. Si la respuesta no aparece en el contexto, responde exactamente:
              "No encontré esa información en los documentos proporcionados."
            4. Responde siempre en español.
            5. Utiliza un lenguaje claro, formal y ordenado.
            6. No mezcles datos de regiones o sedes diferentes.
            7. Cuando existan requisitos o pasos, preséntalos numerados.
            8. Cuando una región tenga varias sedes, enumera cada sede por separado.
            9. Relaciona el horario, dirección, contacto y pago únicamente con
               la sede a la que pertenecen.
            10. Si un campo indica "No indica" o "No reportó", conserva esa precisión.
            11. Puedes utilizar el historial para comprender la conversación,
                pero los datos de la respuesta deben proceder del contexto.
            12. No afirmes que un requisito existe si no figura en los documentos.
            13. No menciones archivos, páginas, documentos recuperados ni fuentes.
            14. Responde únicamente la consulta del usuario.
            """
        ),
        MessagesPlaceholder(variable_name="historial"),
        (
            "human",
            """
Contexto recuperado de los documentos:

{contexto}

Pregunta actual:

{pregunta}
            """
        )
    ]
)

print("Prompt conversacional creado correctamente.")

Prompt conversacional creado correctamente.


In [52]:
def responder_pregunta_con_memoria(
    pregunta: str,
    session_id: str = "sesion-principal"
) -> dict:
    """
    Responde preguntas sobre los PDF conservando el historial
    de conversación de cada sesión.
    """

    if not pregunta or not pregunta.strip():
        return {
            "pregunta_original": pregunta,
            "pregunta_reformulada": "",
            "respuesta": "Debe ingresar una pregunta.",
            "fuentes": [],
            "documentos": []
        }

    pregunta = pregunta.strip()

    # 1. Recuperar el historial de la sesión
    historial_sesion = obtener_historial(session_id)
    mensajes_anteriores = historial_sesion.messages

    # 2. Reformular la pregunta usando el historial
    pregunta_reformulada = reformular_pregunta(
        pregunta=pregunta,
        historial=mensajes_anteriores
    )

    # 3. Buscar en FAISS con la pregunta completa
    documentos_recuperados = retriever.invoke(
        pregunta_reformulada
    )

    # 4. Preparar el contexto
    contexto = formatear_contexto(
        documentos_recuperados
    )

    # 5. Construir el prompt con historial
    mensajes_prompt = prompt_rag_con_memoria.invoke(
        {
            "historial": mensajes_anteriores,
            "contexto": contexto,
            "pregunta": pregunta
        }
    )

    # 6. Generar la respuesta
    resultado = llm.invoke(mensajes_prompt)
    respuesta = resultado.content.strip()

    # 7. Guardar pregunta y respuesta en el historial
    historial_sesion.add_user_message(pregunta)
    historial_sesion.add_ai_message(respuesta)

    # 8. Preparar fuentes sin duplicados
    fuentes = []

    for documento in documentos_recuperados:
        archivo = documento.metadata.get(
            "source",
            "Documento desconocido"
        )

        pagina = documento.metadata.get(
            "pagina_real",
            documento.metadata.get("page", 0) + 1
        )

        fuente = {
            "archivo": archivo,
            "pagina": pagina
        }

        if fuente not in fuentes:
            fuentes.append(fuente)

    return {
        "pregunta_original": pregunta,
        "pregunta_reformulada": pregunta_reformulada,
        "respuesta": respuesta,
        "fuentes": fuentes,
        "documentos": documentos_recuperados,
        "session_id": session_id
    }

In [53]:
def preguntar_al_agente_con_memoria(
    pregunta: str,
    session_id: str = "sesion-principal"
) -> dict:

    resultado = responder_pregunta_con_memoria(
        pregunta=pregunta,
        session_id=session_id
    )

    print("=" * 90)
    print("PREGUNTA ORIGINAL")
    print("=" * 90)
    print(resultado["pregunta_original"])

    print("\n" + "=" * 90)
    print("PREGUNTA UTILIZADA PARA BUSCAR EN FAISS")
    print("=" * 90)
    print(resultado["pregunta_reformulada"])

    print("\n" + "=" * 90)
    print("RESPUESTA")
    print("=" * 90)
    print(resultado["respuesta"])

    print("\n" + "=" * 90)
    print("FUENTES RECUPERADAS")
    print("=" * 90)

    if resultado["fuentes"]:
        for fuente in resultado["fuentes"]:
            print(
                f"- {fuente['archivo']}, "
                f"página {fuente['pagina']}"
            )
    else:
        print("No se recuperaron fuentes.")

    return resultado

In [54]:
almacen_conversaciones.clear()

print("Todas las conversaciones fueron reiniciadas.")

Todas las conversaciones fueron reiniciadas.


In [55]:
def mostrar_historial(
    session_id: str = "sesion-principal"
) -> None:

    historial = obtener_historial(session_id)

    print(f"HISTORIAL DE LA SESIÓN: {session_id}")
    print("=" * 90)

    if not historial.messages:
        print("La conversación está vacía.")
        return

    for mensaje in historial.messages:

        if isinstance(mensaje, HumanMessage):
            rol = "USUARIO"

        elif isinstance(mensaje, AIMessage):
            rol = "ASISTENTE"

        else:
            rol = mensaje.__class__.__name__

        print(f"\n{rol}:")
        print(mensaje.content)

In [56]:
mostrar_historial("prueba-retcc")

HISTORIAL DE LA SESIÓN: prueba-retcc
La conversación está vacía.


In [57]:
def reiniciar_conversacion(
    session_id: str = "sesion-principal"
) -> None:

    if session_id in almacen_conversaciones:
        almacen_conversaciones[session_id].clear()

    print(
        f"Conversación '{session_id}' "
        "reiniciada correctamente."
    )

In [58]:
import gradio as gr

print("Versión de Gradio:", gr.__version__)

Versión de Gradio: 5.49.1


In [59]:
reiniciar_conversacion("prueba-retcc")


Conversación 'prueba-retcc' reiniciada correctamente.


In [60]:
print("Versión de Gradio:", gr.__version__)

Versión de Gradio: 5.49.1


In [81]:
#------------------------------------------
#función para conectar Gradio con el agente
#------------------------------------------
import uuid

In [82]:
def crear_mensaje(rol: str, texto: str) -> dict:
    return {
        "role": rol,
        "content": texto
    }

In [83]:
import html
import traceback
import uuid

from pathlib import Path
import os

# ---------------------------------------------------------
# ARCHIVOS VISUALES DEL FRONTEND
# ---------------------------------------------------------
# Guarda las imágenes dentro de la carpeta "retcc_assets".
# Así evitamos colocar imágenes grandes en formato Base64 dentro del código.
RUTA_ASSETS_RETCC = Path.cwd() / "retcc_assets"
RUTA_ASSETS_RETCC.mkdir(parents=True, exist_ok=True)

# Gradio sirve directamente los archivos de esta carpeta.
try:
    gr.set_static_paths(paths=[str(RUTA_ASSETS_RETCC.resolve())])
except Exception:
    pass

def url_asset_retcc(nombre_archivo: str) -> str:
    ruta = (RUTA_ASSETS_RETCC / nombre_archivo).resolve()
    return f"/gradio_api/file={ruta.as_posix()}"

AVATAR_RETCC_SRC = url_asset_retcc("avatar_retcc.png")
CASCO_RETCC_SRC = url_asset_retcc("casco.svg")



def normalizar_historial_html(historial):
    """
    Convierte cualquier historial previo en una lista simple:
    [{"role": "user|assistant", "content": "..."}]
    """
    resultado = []

    for item in historial or []:

        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            role = item[0]
            content = item[1]

        else:
            continue

        if role not in {"user", "assistant"}:
            continue

        resultado.append(
            {
                "role": role,
                "content": str(content)
            }
        )

    return resultado


def texto_a_html(texto: str) -> str:
    """
    Escapa HTML y conserva saltos de línea.
    También convierte listas Markdown simples en una presentación legible.
    """
    texto = html.escape(str(texto or ""))

    lineas = texto.splitlines()
    partes = []
    lista_abierta = False

    for linea in lineas:
        limpia = linea.strip()

        if limpia.startswith("* "):
            if not lista_abierta:
                partes.append("<ul>")
                lista_abierta = True

            partes.append(
                f"<li>{limpia[2:]}</li>"
            )

        elif limpia.startswith("- "):
            if not lista_abierta:
                partes.append("<ul>")
                lista_abierta = True

            partes.append(
                f"<li>{limpia[2:]}</li>"
            )

        else:
            if lista_abierta:
                partes.append("</ul>")
                lista_abierta = False

            if limpia:
                partes.append(f"<p>{limpia}</p>")
            else:
                partes.append("<br>")

    if lista_abierta:
        partes.append("</ul>")

    return "".join(partes)


def renderizar_chat_html(historial) -> str:
    """
    Renderiza el historial como HTML propio.

    Se evita gr.Chatbot porque su estructura interna estaba siendo
    afectada por estilos y wrappers diferentes entre versiones de Gradio.
    """
    historial = normalizar_historial_html(historial)

    if not historial:
        return """
        <div class="chat-vacio-retcc">
            <span>La conversación aparecerá aquí.</span>
        </div>
        """

    mensajes_html = []

    for mensaje in historial:
        role = mensaje["role"]
        contenido = texto_a_html(mensaje["content"])

        clase = (
            "mensaje-usuario-retcc"
            if role == "user"
            else "mensaje-asistente-retcc"
        )

        etiqueta = (
            "Tú"
            if role == "user"
            else "RetccBob"
        )

        avatar_html = (
            f"""
            <div class="avatar-respuesta-retcc">
                <img src="{AVATAR_RETCC_SRC}" alt="RetccBob">
            </div>
            """
            if role == "assistant"
            else ""
        )

        mensajes_html.append(
            f"""
            <div class="fila-mensaje-retcc {clase}">
                {avatar_html}
                <div class="burbuja-mensaje-retcc">
                    <div class="etiqueta-mensaje-retcc">{etiqueta}</div>
                    <div class="contenido-mensaje-retcc">
                        {contenido}
                    </div>
                </div>
            </div>
            """
        )

    return (
        '<div id="historial-chat-html-retcc">'
        + "".join(mensajes_html)
        + '<div id="fin-chat-retcc"></div>'
        + "</div>"
    )


def procesar_mensaje_html(
    mensaje: str,
    historial_estado: list,
    session_id: str
):
    """
    Envía la consulta al backend y devuelve:
    - input vacío;
    - HTML actualizado;
    - historial actualizado;
    - session_id.
    """

    historial = normalizar_historial_html(historial_estado)

    if not mensaje or not mensaje.strip():
        return (
            "",
            renderizar_chat_html(historial),
            historial,
            session_id
        )

    if not session_id:
        session_id = str(uuid.uuid4())

    pregunta = mensaje.strip()

    historial.append(
        {
            "role": "user",
            "content": pregunta
        }
    )

    try:
        print("\n" + "=" * 90)
        print("CONSULTA RECIBIDA DESDE GRADIO")
        print("=" * 90)
        print("Pregunta:", pregunta)
        print("Session ID:", session_id)

        resultado = responder_pregunta_con_memoria(
            pregunta=pregunta,
            session_id=session_id
        )

        print("Resultado recibido:", resultado)

        respuesta = resultado.get(
            "respuesta",
            "No se pudo generar una respuesta."
        )

        historial.append(
            {
                "role": "assistant",
                "content": respuesta
            }
        )

    except Exception:
        print("\n" + "=" * 90)
        print("ERROR COMPLETO DEL CHATBOT")
        print("=" * 90)
        traceback.print_exc()

        historial.append(
            {
                "role": "assistant",
                "content": (
                    "No fue posible procesar la consulta. "
                    "Revise el error mostrado en Google Colab."
                )
            }
        )

    return (
        "",
        renderizar_chat_html(historial),
        historial,
        session_id
    )


def ejecutar_consulta_rapida_html(
    pregunta: str,
    historial_estado: list,
    session_id: str
):
    return procesar_mensaje_html(
        mensaje=pregunta,
        historial_estado=historial_estado,
        session_id=session_id
    )


def nueva_conversacion_html(session_id: str):
    """
    Limpia memoria e historial visual.
    """
    if session_id:
        try:
            reiniciar_conversacion(session_id)
        except Exception as error:
            print("No se pudo reiniciar la conversación:", repr(error))

    nuevo_session_id = str(uuid.uuid4())
    historial = []

    return (
        renderizar_chat_html(historial),
        historial,
        nuevo_session_id,
        ""
    )


In [85]:
# ---------------------------------------------------------
# FUNCIÓN PARA INICIAR UNA CONVERSACIÓN NUEVA
# ---------------------------------------------------------

def nueva_conversacion(session_id: str):
    """
    Limpia la memoria de la sesión actual y crea una sesión nueva.
    La bienvenida permanece visible en la parte superior de la interfaz.
    """

    if session_id:
        try:
            reiniciar_conversacion(session_id)
        except Exception as error:
            print(
                "No se pudo reiniciar la conversación:",
                repr(error)
            )

    nuevo_session_id = str(uuid.uuid4())

    return [], nuevo_session_id, ""

In [86]:
def crear_mensaje(rol: str, texto: str) -> dict:
    return {
        "role": rol,
        "content": texto
    }

In [96]:
# ---------------------------------------------------------
# CELDA 58: LAYOUT FINAL ESTABLE
# Sidebar fija + conversación visible + input fijo
# ---------------------------------------------------------

CSS_RETCC = """
:root {
    --rojo: #c90008;
    --rojo-oscuro: #9b0005;
    --fondo: #faf8f7;
    --blanco: #ffffff;
    --borde: #edb8b4;
    --texto: #252525;
}

* {
    box-sizing: border-box !important;
}

html,
body,
gradio-app,
.gradio-container {
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: var(--fondo) !important;
    color-scheme: light !important;
    font-family: Inter, "Segoe UI", Arial, sans-serif !important;
}

footer {
    display: none !important;
}

/* =====================================================
   ESTRUCTURA GENERAL
   ===================================================== */

#layout-retcc {
    width: 100% !important;
    height: 100vh !important;
    height: 100dvh !important;
    min-height: 0 !important;
    display: flex !important;
    flex-wrap: nowrap !important;
    gap: 0 !important;
    margin: 0 !important;
    overflow: hidden !important;
}

#panel-lateral {
    position: relative !important;
    flex: 0 0 290px !important;
    width: 308px !important;
    min-width: 308px !important;
    max-width: 308px !important;
    height: 100% !important;
    min-height: 0 !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: #ffffff !important;
    border-right: 1px solid var(--borde) !important;
}

#zona-principal {
    flex: 1 1 auto !important;
    width: auto !important;
    min-width: 0 !important;
    height: 100% !important;
    min-height: 0 !important;
    display: grid !important;
    grid-template-rows: 74px 32px minmax(0, 1fr) !important;
    gap: 0 !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: var(--fondo) !important;
}

/* =====================================================
   SIDEBAR EXACTA
   ===================================================== */

.sidebar-retcc {
    position: relative;
    width: 100%;
    height: 100%;
    background: #ffffff;
}

.sidebar-superior {
    padding: 28px 25px 16px;
}

.titulo-recursos {
    margin: 0 0 19px;
    color: #a50005 !important;
    font-size: 14px;
    font-weight: 800;
    letter-spacing: .15px;
}

.recurso-retcc {
    min-height: 60px;
    display: flex;
    align-items: center;
    gap: 14px;
    padding: 10px 14px;
    color: #282828 !important;
    text-decoration: none !important;
    border-radius: 8px;
    font-size: 15px;
    font-weight: 500;
}

.recurso-retcc span {
    color: #282828 !important;
    opacity: 1 !important;
}

.recurso-retcc:hover {
    background: #f7f2f1;
}

.recurso-icono {
    width: 25px;
    min-width: 25px;
    color: #505050 !important;
    text-align: center;
    font-size: 21px;
}

.sidebar-inferior {
    position: absolute;
    left: 0;
    right: 0;
    bottom: 18px;
    padding: 25px 25px 0;
    border-top: 1px solid var(--borde);
    background: #ffffff;
}

.tarjeta-ayuda {
    min-height: 164px;
    padding: 18px 17px 66px;
    margin: 0 0 26px;
    background: #fde8e6;
    border-radius: 10px;
}

.tarjeta-ayuda h3 {
    margin: 0 0 10px;
    color: #8d0004 !important;
    font-size: 14px;
    font-weight: 800;
}

.tarjeta-ayuda p {
    margin: 0;
    color: #5b4542 !important;
    font-size: 13px;
    line-height: 1.55;
}

.usuario-retcc {
    min-height: 56px;
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 3px 8px;
}

.usuario-avatar {
    width: 43px;
    height: 43px;
    display: grid;
    place-items: center;
    color: #444444;
    background: #ece9e7;
    border-radius: 12px;
    font-size: 20px;
}

.usuario-nombre {
    color: #222222 !important;
    font-size: 14px;
    font-weight: 800;
    line-height: 1.15;
}

.usuario-rol {
    margin-top: 2px;
    color: #6c6c6c !important;
    font-size: 12px;
    line-height: 1.15;
}

#boton-soporte {
    position: absolute !important;
    left: 43px !important;
    bottom: 109px !important;
    z-index: 20 !important;
    width: 222px !important;
    min-width: 222px !important;
    max-width: 222px !important;
    min-height: 40px !important;
    margin: 0 !important;
    color: #ffffff !important;
    background: var(--rojo) !important;
    border: 0 !important;
    border-radius: 5px !important;
    font-size: 13px !important;
    font-weight: 800 !important;
}

#boton-nuevo {
    display: none !important;
}


.logo-retcc {
    width: 44px !important;
    height: 44px !important;
    display: grid !important;
    place-items: center !important;
    overflow: hidden !important;
    background: #ffffff !important;
    border-radius: 11px !important;
}

.logo-retcc img {
    width: 36px !important;
    height: 36px !important;
    display: block !important;
    object-fit: contain !important;
}

/* =====================================================
   CABECERA Y ESTADO
   ===================================================== */

#cabecera-wrap {
    grid-row: 1 !important;
    width: 100% !important;
    height: 74px !important;
    min-height: 74px !important;
    padding: 0 !important;
    overflow: hidden !important;
}

#estado-wrap {
    grid-row: 2 !important;
    width: 100% !important;
    height: 32px !important;
    min-height: 32px !important;
    padding: 0 !important;
    overflow: hidden !important;
}

.cabecera-retcc {
    width: 100%;
    height: 74px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 28px;
    color: #ffffff;
    background: var(--rojo);
    box-shadow: 0 2px 8px rgba(80, 0, 0, .14);
}

.cabecera-marca {
    display: flex;
    align-items: center;
    gap: 13px;
}

.logo-retcc {
    width: 42px;
    height: 42px;
    display: grid;
    place-items: center;
    padding: 5px;
    background: #ffffff;
    border-radius: 12px;
}

.casco-retcc {
    width: 32px;
    height: 32px;
}

.cabecera-titulo {
    margin: 0;
    color: #ffffff !important;
    font-size: 20px;
    font-weight: 800;
}

.cabecera-info {
    color: #ffffff !important;
    font-size: 24px;
}

.estado-retcc {
    width: 100%;
    height: 32px;
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 16px;
    color: #555555;
    background: #f2efee;
    border-bottom: 1px solid var(--borde);
    font-size: 12px;
}

.estado-retcc strong {
    color: var(--rojo);
}

.barra-estado {
    width: 132px;
    height: 6px;
    overflow: hidden;
    background: #e9c4c1;
    border-radius: 999px;
}

.barra-estado span {
    display: block;
    width: 22%;
    height: 100%;
    background: var(--rojo);
}

/* =====================================================
   CONTENIDO CENTRAL
   ===================================================== */

#contenido-retcc {
    grid-row: 3 !important;
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
    display: grid !important;
    grid-template-rows: minmax(0, 1fr) auto !important;
    gap: 0 !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: var(--fondo) !important;
}

#conversacion-scroll {
    grid-row: 1 !important;
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
    display: block !important;
    padding: 18px 28px 16px !important;
    overflow-x: hidden !important;
    overflow-y: auto !important;
    background: var(--fondo) !important;
    scrollbar-width: thin;
    scrollbar-color: #cfa29e transparent;
}

#conversacion-scroll::-webkit-scrollbar {
    width: 8px;
}

#conversacion-scroll::-webkit-scrollbar-thumb {
    background: #cfa29e;
    border-radius: 999px;
}

#zona-entrada {
    grid-row: 2 !important;
    width: 100% !important;
    min-height: 78px !important;
    padding: 8px 28px 7px !important;
    background: var(--fondo) !important;
    border-top: 1px solid var(--borde) !important;
}

/* =====================================================
   BLOQUES DE BIENVENIDA
   ===================================================== */

#bloque-bienvenida,
#bloque-informativo,
.consultas-grid,
#chat-html-retcc {
    width: min(100%, 760px) !important;
    margin-left: auto !important;
    margin-right: auto !important;
}

.fila-asistente {
    display: grid;
    grid-template-columns: 42px minmax(0, 1fr);
    gap: 16px;
    align-items: start;
}

.icono-asistente {
    width: 42px;
    height: 42px;
    display: grid;
    place-items: center;
    overflow: hidden;
    background: #ffffff;
    border: 2px solid var(--rojo);
    border-radius: 13px;
    box-shadow: 0 2px 7px rgba(85, 0, 0, .15);
}

.icono-asistente img {
    width: 100%;
    height: 100%;
    object-fit: cover;
    object-position: center 18%;
}

.burbuja-bienvenida {
    padding: 17px 19px;
    color: var(--texto) !important;
    background: #ffffff;
    border: 1px solid var(--borde);
    border-radius: 11px;
    font-size: 15px;
    line-height: 1.45;
}

.burbuja-bienvenida * {
    color: var(--texto) !important;
}

.consultas-grid {
    width: min(calc(100% - 58px), 702px) !important;
    margin-top: 10px !important;
    transform: translateX(29px);
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 9px !important;
}

.boton-rapido {
    width: auto !important;
    min-width: 0 !important;
    min-height: 38px !important;
    flex: 0 0 auto !important;
    padding: 7px 16px !important;
    color: var(--rojo) !important;
    background: #ffffff !important;
    border: 1px solid var(--rojo) !important;
    border-radius: 12px !important;
    box-shadow: none !important;
    font-size: 12px !important;
    font-weight: 750 !important;
}

#bloque-informativo {
    margin-top: 12px !important;
}

.tarjeta-informativa {
    overflow: hidden;
    min-height: 245px;
    background: #ffffff;
    border: 1px solid var(--borde);
    border-radius: 10px;
}

.imagen-informativa {
    height: 160px;
    min-height: 160px;
    display: flex;
    align-items: end;
    padding: 18px 20px;
    color: #ffffff;
    background:
        linear-gradient(0deg, rgba(70, 0, 0, .72), rgba(120, 0, 0, .12)),
        url("https://lh3.googleusercontent.com/aida-public/AB6AXuAKUeXdi1JWzDkEgAfqRIrSYHrMc6M9PsnNkvXkc5_jWALheiEB6LZaQK3YWuqE8bbDHRofbA1cE-C7cFWmV9y1XHz-ip2mes16dRQ_XGnFBsijM8ibFuWUA2KB5Re9DW_uqd2bfyFcGbJG0Gauyb8_BQDvUcWHoQRsiAkZtqDiSy_77EpQgBl5S3tgCpaZ7ltLS7fOlKmyi_xzvcK0t1AXFPTs8K3hxFxo_Uj31uZUGvBlR2D5htoodA");
    background-size: cover;
    background-position: center 48%;
}

.imagen-informativa strong {
    color: #ffffff !important;
    font-size: 17px;
}

.contenido-informativo {
    padding: 12px 18px 13px;
    color: #333333;
    font-size: 12px;
    line-height: 1.4;
}

.contenido-informativo * {
    color: #333333 !important;
}

.lista-informativa {
    display: grid;
    gap: 5px;
}

.lista-informativa span::before {
    content: "✓";
    display: inline-grid;
    place-items: center;
    width: 14px;
    height: 14px;
    margin-right: 6px;
    color: var(--rojo);
    border: 1.3px solid var(--rojo);
    border-radius: 50%;
    font-size: 8px;
    font-weight: 900;
}

/* =====================================================
   CHAT HTML
   ===================================================== */

#chat-html-retcc {
    margin-top: 14px !important;
    padding-bottom: 6px !important;
}

#chat-html-retcc .html-container,
#chat-html-retcc .prose {
    width: 100% !important;
    max-width: none !important;
    padding: 0 !important;
    margin: 0 !important;
    background: transparent !important;
}

#historial-chat-html-retcc {
    width: 100%;
    display: flex;
    flex-direction: column;
    gap: 12px;
}

.chat-vacio-retcc {
    min-height: 20px;
    color: #8a8a8a;
    text-align: center;
    font-size: 12px;
}

.fila-mensaje-retcc {
    width: 100%;
    display: flex;
}

.mensaje-usuario-retcc {
    justify-content: flex-end;
}

.mensaje-asistente-retcc {
    justify-content: flex-start;
    align-items: flex-start;
    gap: 12px;
}

.avatar-respuesta-retcc {
    flex: 0 0 42px;
    width: 42px;
    height: 42px;
    display: grid;
    place-items: center;
    overflow: hidden;
    background: #ffffff;
    border: 2px solid var(--rojo);
    border-radius: 13px;
    box-shadow: 0 2px 7px rgba(85, 0, 0, .15);
}

.avatar-respuesta-retcc img {
    width: 100%;
    height: 100%;
    object-fit: cover;
    object-position: center 18%;
}

.burbuja-mensaje-retcc {
    width: fit-content;
    min-width: 150px;
    max-width: 78%;
    padding: 12px 15px;
    border-radius: 14px;
    box-shadow: 0 1px 3px rgba(50, 0, 0, .05);
}

.mensaje-usuario-retcc .burbuja-mensaje-retcc {
    color: #ffffff;
    background: var(--rojo);
    border-bottom-right-radius: 4px;
}

.mensaje-asistente-retcc .burbuja-mensaje-retcc {
    color: #111111 !important;
    background: #ffffff;
    border: 1px solid var(--borde);
    border-bottom-left-radius: 4px;
}

.mensaje-asistente-retcc .etiqueta-mensaje-retcc,
.mensaje-asistente-retcc .contenido-mensaje-retcc,
.mensaje-asistente-retcc .contenido-mensaje-retcc * {
    color: #111111 !important;
    opacity: 1 !important;
}

.etiqueta-mensaje-retcc {
    margin-bottom: 5px;
    font-size: 10px;
    font-weight: 800;
    opacity: .8;
    text-transform: uppercase;
    letter-spacing: .4px;
}

.contenido-mensaje-retcc {
    font-size: 13px;
    line-height: 1.5;
    overflow-wrap: anywhere;
}

.contenido-mensaje-retcc p {
    margin: 0 0 8px;
}

.contenido-mensaje-retcc p:last-child {
    margin-bottom: 0;
}

/* =====================================================
   INPUT
   ===================================================== */

#fila-entrada {
    width: min(100%, 900px) !important;
    min-height: 58px !important;
    margin: 0 auto !important;
    display: flex !important;
    align-items: center !important;
    gap: 7px !important;
    padding: 6px !important;
    background: #ffffff !important;
    border: 1px solid var(--borde) !important;
    border-radius: 15px !important;
}

#entrada-mensaje {
    flex: 1 1 auto !important;
    min-width: 0 !important;
}

#entrada-mensaje,
#entrada-mensaje > div {
    background: #ffffff !important;
    border: 0 !important;
    box-shadow: none !important;
}

#entrada-mensaje textarea {
    min-height: 44px !important;
    max-height: 76px !important;
    padding: 11px 12px !important;
    color: #292929 !important;
    background: #ffffff !important;
    border: 0 !important;
    box-shadow: none !important;
    font-size: 14px !important;
}

#boton-enviar {
    flex: 0 0 48px !important;
    min-width: 48px !important;
    max-width: 48px !important;
    height: 44px !important;
    padding: 0 !important;
    color: #ffffff !important;
    background: var(--rojo) !important;
    border: 0 !important;
    border-radius: 10px !important;
}

.pie-retcc {
    margin-top: 3px;
    color: #707070;
    text-align: center;
    font-size: 8px;
    letter-spacing: .6px;
    text-transform: uppercase;
}

/* =====================================================
   RESPONSIVE
   ===================================================== */

@media (max-width: 900px) {
    #panel-lateral {
        display: none !important;
    }

    #conversacion-scroll {
        padding: 10px 12px 9px !important;
    }

    #zona-entrada {
        padding: 6px 12px 5px !important;
    }

    .burbuja-mensaje-retcc {
        max-width: 92%;
    }
}


/* =====================================================
   AJUSTE FINAL DE LA BARRA LATERAL
   Replica la distribución del modelo compartido.
   ===================================================== */

#panel-lateral,
#panel-lateral > div,
#panel-lateral .form,
#panel-lateral .wrap {
    height: 100% !important;
    min-height: 0 !important;
    padding: 0 !important;
    margin: 0 !important;
    gap: 0 !important;
    overflow: hidden !important;
    position: relative !important;
    background: #ffffff !important;
}

#panel-lateral .html-container {
    position: absolute !important;
    inset: 0 !important;
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
    padding: 0 !important;
    margin: 0 !important;
    overflow: hidden !important;
    border: 0 !important;
    background: transparent !important;
}

.sidebar-retcc {
    position: absolute !important;
    inset: 0 !important;
    display: flex !important;
    flex-direction: column !important;
    width: 100% !important;
    height: 100% !important;
    min-height: 0 !important;
    background: #ffffff !important;
}

.sidebar-superior {
    flex: 0 0 auto !important;
    padding: 28px 25px 16px !important;
}

.sidebar-inferior {
    position: absolute !important;
    left: 0 !important;
    right: 0 !important;
    bottom: 18px !important;
    padding: 25px 25px 0 !important;
    margin: 0 !important;
    border-top: 1px solid var(--borde) !important;
    background: #ffffff !important;
}

.tarjeta-ayuda {
    width: 100% !important;
    min-height: 164px !important;
    padding: 18px 17px 66px !important;
    margin: 0 0 26px !important;
    background: #fde8e6 !important;
    border-radius: 10px !important;
}

.usuario-retcc {
    width: 100% !important;
    min-height: 56px !important;
    padding: 3px 8px !important;
    margin: 0 !important;
}

#boton-soporte {
    position: absolute !important;
    left: 43px !important;
    bottom: 109px !important;
    z-index: 50 !important;
    width: 222px !important;
    min-width: 222px !important;
    max-width: 222px !important;
    height: 40px !important;
    min-height: 40px !important;
    padding: 0 16px !important;
    margin: 0 !important;
    color: #ffffff !important;
    background: var(--rojo) !important;
    border: 0 !important;
    border-radius: 5px !important;
    box-shadow: none !important;
    font-size: 13px !important;
    font-weight: 800 !important;
}

#boton-soporte > button {
    width: 100% !important;
    height: 100% !important;
    min-height: 40px !important;
    padding: 0 !important;
    margin: 0 !important;
    color: #ffffff !important;
    background: var(--rojo) !important;
    border: 0 !important;
    border-radius: 5px !important;
    box-shadow: none !important;
    font-weight: 800 !important;
}

#boton-nuevo {
    display: none !important;
}

/* Imágenes externas: ya no se incrustan en Base64. */
.icono-asistente img,
.avatar-respuesta-retcc img,
.logo-retcc img {
    display: block !important;
    object-fit: contain !important;
}

"""

In [97]:
# ---------------------------------------------------------
# CELDA 59: INTERFAZ FINAL CON LAYOUT FIJO
# ---------------------------------------------------------

import uuid
import gradio as gr


with gr.Blocks(
    title="Chat RETCC Bob",
    css=CSS_RETCC,
    fill_height=True,
    theme=gr.themes.Soft(
        primary_hue="red",
        neutral_hue="gray"
    )
) as interfaz_retcc:

    session_id = gr.State(value=str(uuid.uuid4()))
    historial_estado = gr.State(value=[])

    with gr.Row(elem_id="layout-retcc"):

        with gr.Column(
            scale=0,
            min_width=308,
            elem_id="panel-lateral"
        ):
            gr.HTML(
                """
                <aside class="sidebar-retcc">
                    <div class="sidebar-superior">
                        <h2 class="titulo-recursos">RECURSOS OFICIALES</h2>

                        <a class="recurso-retcc" href="#">
                            <span class="recurso-icono">▤</span>
                            <span>Guía de Inscripción</span>
                        </a>

                        <a class="recurso-retcc" href="#">
                            <span class="recurso-icono">♢</span>
                            <span>Reglamento RETCC</span>
                        </a>

                        <a class="recurso-retcc" href="#">
                            <span class="recurso-icono">▣</span>
                            <span>Cronograma 2024</span>
                        </a>
                    </div>

                    <div class="sidebar-inferior">
                        <div class="tarjeta-ayuda">
                            <h3>¿Necesitas Ayuda?</h3>
                            <p>
                                Nuestro equipo de soporte está disponible
                                de L-V 8am-5pm.
                            </p>
                        </div>

                        <div class="usuario-retcc">
                            <div class="usuario-avatar">♙</div>
                            <div>
                                <div class="usuario-nombre">Juan Pérez</div>
                                <div class="usuario-rol">Trabajador Civil</div>
                            </div>
                        </div>
                    </div>
                </aside>
                """
            )

            boton_soporte = gr.Button(
                "Contactar Soporte",
                elem_id="boton-soporte"
            )

            boton_nuevo = gr.Button(
                "Nueva conversación",
                elem_id="boton-nuevo"
            )

        with gr.Column(
            scale=1,
            min_width=0,
            elem_id="zona-principal"
        ):

            gr.HTML(
                f"""
                <header class="cabecera-retcc">
                    <div class="cabecera-marca">

<div class="logo-retcc" aria-label="Casco de construcción">
    <img src="{CASCO_RETCC_SRC}" alt="Casco de construcción">
</div>

                        <h1 class="cabecera-titulo">Chat RETCC BOB tu Asistente Virtual</h1>
                    </div>
                    <div class="cabecera-info">ⓘ</div>
                </header>
                """,
                elem_id="cabecera-wrap"
            )

            gr.HTML(
                """
                <div class="estado-retcc">
                    <div>
                        Estado de Consulta:
                        <strong>Iniciando</strong>
                    </div>
                    <div class="barra-estado">
                        <span></span>
                    </div>
                </div>
                """,
                elem_id="estado-wrap"
            )

            with gr.Column(elem_id="contenido-retcc"):

                with gr.Column(elem_id="conversacion-scroll"):

                    gr.HTML(
                        f"""
                        <section id="bloque-bienvenida">
                            <div class="fila-asistente">
                                <div class="icono-asistente"><img src="{AVATAR_RETCC_SRC}" alt="Asistente virtual de construcción Civil"></div>

                                <div class="burbuja-bienvenida">
                                    ¡Hola! Soy RetccBob tu asistente virtual del
                                    <strong>
                                        Registro Nacional de Trabajadores de
                                        Construcción Civil (RETCC).
                                    </strong>

                                    <br><br>

                                    Estoy aquí para ayudarte con información
                                    oficial sobre trámites, requisitos y
                                    renovaciones de tu carné.
                                    ¿En qué puedo apoyarte hoy?
                                </div>
                            </div>
                        </section>
                        """
                    )

                    with gr.Row(elem_classes=["consultas-grid"]):
                        btn_requisitos = gr.Button(
                            "Requisitos para inscripción",
                            elem_classes=["boton-rapido"]
                        )
                        btn_renovacion = gr.Button(
                            "¿Cómo renovar mi carné?",
                            elem_classes=["boton-rapido"]
                        )
                        btn_costo = gr.Button(
                            "Costo del trámite en Lima Metropolitana",
                            elem_classes=["boton-rapido"]
                        )

                    gr.HTML(
                        """
                        <section id="bloque-informativo">
                            <div class="fila-asistente">
                                <div class="icono-asistente"
                                     style="background:#c90008;color:white;border:none;">
                                    ⓘ
                                </div>

                                <div class="tarjeta-informativa">
                                    <div class="imagen-informativa">
                                        <strong>Información Importante</strong>
                                    </div>

                                    <div class="contenido-informativo">
                                        <p>
                                            Recuerde que la inscripción en el RETCC
                                            es obligatoria para todo trabajador que
                                            realice labores en construcción civil.
                                        </p>

                                        <div class="lista-informativa">
                                            <span>Validez a nivel nacional</span>
                                            <span>Vigencia de 2 años</span>
                                        </div>
                                    </div>
                                </div>
                            </div>
                        </section>
                        """
                    )

                    chat_html = gr.HTML(
                        value=renderizar_chat_html([]),
                        elem_id="chat-html-retcc"
                    )

                with gr.Column(elem_id="zona-entrada"):
                    with gr.Row(elem_id="fila-entrada"):
                        entrada_mensaje = gr.Textbox(
                            placeholder="Escribe tu consulta aquí...",
                            show_label=False,
                            lines=1,
                            max_lines=4,
                            elem_id="entrada-mensaje",
                            scale=10,
                            container=False
                        )

                        boton_enviar = gr.Button(
                            "➤",
                            elem_id="boton-enviar",
                            scale=0
                        )

                    gr.HTML(
                        """
                        <div class="pie-retcc">
                            Copyright © [Desarrollado by Arturo Guerrero - 2026]
                        </div>
                        """
                    )

    boton_enviar.click(
        fn=procesar_mensaje_html,
        inputs=[entrada_mensaje, historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    entrada_mensaje.submit(
        fn=procesar_mensaje_html,
        inputs=[entrada_mensaje, historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    boton_nuevo.click(
        fn=nueva_conversacion_html,
        inputs=[session_id],
        outputs=[chat_html, historial_estado, session_id, entrada_mensaje],
        show_progress="hidden"
    )

    btn_requisitos.click(
        fn=lambda historial, sesion: ejecutar_consulta_rapida_html(
            "¿Cuáles son los requisitos para inscribirme en el RETCC?",
            historial,
            sesion
        ),
        inputs=[historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    btn_renovacion.click(
        fn=lambda historial, sesion: ejecutar_consulta_rapida_html(
            "¿Cómo puedo renovar mi carné RETCC?",
            historial,
            sesion
        ),
        inputs=[historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    btn_costo.click(
        fn=lambda historial, sesion: ejecutar_consulta_rapida_html(
            "¿Cuál es el costo del trámite del RETCC?",
            historial,
            sesion
        ),
        inputs=[historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    btn_estado.click(
        fn=lambda historial, sesion: ejecutar_consulta_rapida_html(
            "¿Cómo puedo verificar el estado de mi trámite RETCC?",
            historial,
            sesion
        ),
        inputs=[historial_estado, session_id],
        outputs=[entrada_mensaje, chat_html, historial_estado, session_id],
        show_progress="hidden"
    )

    boton_soporte.click(
        fn=lambda: "Necesito orientación adicional sobre mi trámite RETCC.",
        inputs=None,
        outputs=[entrada_mensaje],
        show_progress="hidden"
    )

In [ ]:
def url_asset_retcc(nombre_archivo: str) -> str:
    ruta = (RUTA_ASSETS_RETCC / nombre_archivo).resolve()
    return f"/gradio_api/file={ruta.as_posix()}"

CASCO_RETCC_SRC = url_asset_retcc("casco.svg")

# Ruta física para el favicon
CASCO_RETCC_FAVICON = (
    RUTA_ASSETS_RETCC / "casco.svg"
).resolve()


interfaz_retcc.launch(
    share=True,
    inline=False,
    debug=True,
    show_error=True,
    favicon_path=str(CASCO_RETCC_FAVICON)
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ea3a05c6b6b95472d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

CONSULTA RECIBIDA DESDE GRADIO
Pregunta: ¿Cuál es el costo del trámite del RETCC?
Session ID: 2f2b708b-a118-4734-9622-98c48e2d00a0
Resultado recibido: {'pregunta_original': '¿Cuál es el costo del trámite del RETCC?', 'pregunta_reformulada': '¿Cuál es el costo del trámite del RETCC?', 'respuesta': 'El costo del trámite de duplicado del carné del RETCC será fijado por la entidad competente en cada dirección o gerencia regional de trabajo y promoción del empleo, conforme al marco normativo vigente. Para solicitarlo, se requiere el número de operación del pago de la respectiva tasa por d

In [94]:
try:
    interfaz_retcc.close()
except Exception:
    pass

Closing server running on port: 7860
